### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="heart_disease_hungary",
    dataset_year="1989",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C52P4X",
    download_description="""
Get the UCI data.

wget https://archive.ics.uci.edu/static/public/45/heart+disease.zip && unzip heart+disease.zip processed.hungarian.data && rm heart+disease.zip && mkdir -p local-data-warehouse/heart_disease_hungary && mv processed.hungarian.data local-data-warehouse/heart_disease_hungary/
""",
    # References
    academic_reference_bibtex="""@article{detrano1989international,
  title={International application of a new probability algorithm for the diagnosis of coronary artery disease},
  author={Detrano, Robert and Janosi, Andras and Steinbrunn, Walter and Pfisterer, Matthias and Schmid, Johann-Jakob and Sandhu, Sarbjit and Guppy, Kern H and Lee, Stella and Froelicher, Victor},
  journal={The American journal of cardiology},
  volume={64},
  number={5},
  pages={304--310},
  year={1989},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="detrano1989international",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the processed version and the subset of 14 attributes used in the study and clinical practice.

- We encode missing values as np.nan instead of "?".
- We make the target binary (0=no heart disease, 1=heart disease). This follows the original study in attempting to distinguish presence (values 1,2,3,4) from absence (value 0).
- The data has one naturally occurring duplicate, which we do not drop.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="heart_disease_diagnosis",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="heart_disease_diagnosis",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "num"
]

df = pd.read_csv(dataset_mold.path / "processed.hungarian.data", header=None, names=columns)
print("Loaded data shape:", df.shape)

df = df.replace("?", np.nan)
df[["ca", "trestbps", "thalach", "chol"]] = df[["ca", "trestbps", "thalach", "chol"]].astype(float)
# Make target
df["heart_disease_diagnosis"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])

as_cat_type = ["thal", "slope", "exang", "restecg", "fbs", "cp", "sex", "heart_disease_diagnosis"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (294, 14)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 294
Columns: 14
Use sampling: False (sample size: 294)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['chol', 'thalach', 'age', 'trestbps', 'oldpeak', 'cp', 'slope', 'thal', 'restecg', 'fbs']
Rows remaining as candidates after top-10 filter: 2 (of 294)

#### Duplicate Report
Total duplicate rows: 1 (0.34% of dataset)
Duplicate rows ignoring target: 1 (0.34% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,heart_disease_diagnosis
0,39,1,2,120.0,204.0,0,0,145.0,0,0.0,NaN,NaN,NaN,0
1,34,0,2,130.0,161.0,0,0,190.0,0,0.0,NaN,NaN,NaN,0
2,54,0,2,160.0,312.0,0,0,130.0,0,0.0,NaN,NaN,NaN,0
3,47,0,4,120.0,205.0,0,0,98.0,1,2.0,2,NaN,6,1
4,52,1,4,112.0,342.0,0,1,96.0,1,1.0,2,NaN,NaN,1


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,thal,category,266.0,90.48,3.0,"7, 6, 3"
1,slope,category,190.0,64.63,3.0,"2, 1, 3"
2,fbs,category,8.0,2.72,2.0,"0, 1"
3,restecg,category,1.0,0.34,3.0,"0, 1, 2"
4,exang,category,1.0,0.34,2.0,"0, 1"
5,sex,category,0.0,0.00,2.0,"1, 0"
6,cp,category,0.0,0.00,4.0,"4, 2, 3, 1"
7,heart_disease_diagnosis,category,0.0,0.00,2.0,"0, 1"
8,ca,float64,291.0,98.98,1.0,0.0
9,chol,float64,23.0,7.82,153.0,"230.0, 246.0, 275.0, 263.0, 215.0, 224.0, 260.0, 238.0, 196.0, 211.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,294.0,47.826531,7.811812,28.0,66.0
trestbps,293.0,132.583618,17.626568,92.0,200.0
chol,271.0,250.848708,67.657711,85.0,603.0
thalach,293.0,139.129693,23.589749,82.0,190.0
oldpeak,294.0,0.586054,0.908648,0.0,5.0
ca,3.0,0.000000,0.000000,0.0,0.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                  rank                    
cp                      1        4    123  41.84
                        2        2    106  36.05
                        3        3     54  18.37
                        4        1     11   3.74
exang                   1        0    204  69.39
                        2        1     89  30.27
                        3     <NA>      1   0.34
fbs                     1        0    266  90.48
                        2        1     20   6.80
                        3     <NA>      8   2.72
heart_disease_diagnosis 1        0    188  63.95
                        2        1    106  36.05
restecg                 1        0    235  79.93
                        2        1     52  17.69
                        3        2      6   2.04
                        4     <NA>      1   0.34
sex                     1        1    213  72.45
                        2        0     81  27.55
slope                   1     <NA>    190  64.63
                        2        2     91  30.95
                        3        1     12   4.08
                        4        3      1   0.34
thal                    1     <NA>    266  90.48
                        2        7     11   3.74
                        3        6     10   3.40
                        4        3      7   2.38

In [8]:
# Target Distribution
target_df

,count,pct
heart_disease_diagnosis,,
0,188,63.95
1,106,36.05


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to heart_disease_hungary/019d5dcd-8bb5-768b-afbd-4ba72bb25ee6


019d5dcd-8bb5-768b-afbd-4ba72bb25ee6
e68fb46dfb2653477659f99805a923fb619eaa6abb71ea011135d3b2a957406a
